<div align="center">

# PPR-opt

**Design a PPR protein binding a specific RNA sequence of your choice**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/PPROPT_designer.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr/blob/main/LICENSE)
[![Tests](https://img.shields.io/badge/tests-1195%20passing-1a7f5a.svg)](https://github.com/SyedZainAliShah/clippr)

**CLiPPR - iGEM Marburg 2026**

</div>

---

**P**entatrico**p**eptide **R**epeat (PPR) proteins are build from ~31-residue repeats, and **each
repeat reads exactly one RNA base** through two specificity residues:

| 5th + last residue | reads |     | 5th + last residue | reads |
|:---:|:---:|---|:---:|:---:|
| `T` `N` | **A** |  | `T` `D` | **G** |
| `N` `N` | **C** |  | `N` `D` | **U** |

T = threonine
N = asparagine
D = aspartate

Therefore, a RNA sequence of 9 bases would result in a PPR protein of 9 repeats.
This tool will give you a synthesisable coding sequence, a Golden Gate
assembly plan, and the ready-to-order fragments. 

**Four steps.**
Run **0 · Setup** once, edit **1**, then run **2**, **3** and **4** in order.
Everything under *Optional extras* can be skipped - open one when you want that particular check.

The code is hidden behind the forms. You can use a cell's ⋮ **menu → Form → Show code** to read it.

> **You want to design a flexible toolboy instead of a single PPR?**
That is a different workflow, so it has its own notebook here:
> [PPROPT_inventory.ipynb](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/PPROPT_inventory.ipynb).

> ##### Disclaimer:
> **Predicted fidelity** comes from published ligation-count matrices (Pryor *et al.* 2020). **QC** is a common sequence-complexity
> check, this might vary depending on the supplier of your choice. As of now only one synthetic PPR designed for *Chlamydomonas reinhardtii* has been transformed and tested.

## 0 · Setup

Run once - installs PPR-opt from GitHub, takes about a minute.

In [ ]:
#@title Setup - install PPR-opt {display-mode: "form"}
#@markdown Installs PPR-opt or updates it when this notebook is newer than the copy you
#@markdown already have. Safe to re-run. Will take about a minute the first time, seconds after that.
#@markdown
#@markdown **If it fails:** run it again - a dropped network call is the usual cause. If it
#@markdown reports that it updated, use **Runtime → Restart session** and run it once more:
#@markdown the old copy stays loaded in memory until you do. Every cell below needs this one
#@markdown to have succeeded, since do not skip past a red error here.
REPO = "SyedZainAliShah/clippr"

import importlib.metadata as _md, json as _json, sys, urllib.request


def _installed():
    """(kind, commit) for the copy already on this machine, without importing it."""
    try:
        info = _json.loads(_md.distribution("ppropt").read_text("direct_url.json") or "{}")
    except Exception as exc:
        return ("absent", None) if isinstance(exc, _md.PackageNotFoundError) else ("other", None)
    if "vcs_info" in info:
        return "github", info["vcs_info"].get("commit_id")
    if "dir_info" in info or str(info.get("url", "")).startswith("file:"):
        return "local", None
    return "other", None


def _latest():
    """The current main commit, or None when GitHub cannot be reached."""
    try:
        req = urllib.request.Request(f"https://api.github.com/repos/{REPO}/commits/main",
                                     headers={"Accept": "application/vnd.github.sha"})
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.read().decode().strip()
    except Exception:
        return None


# A bare `import ppropt` succeeding is not evidence that the copy is current. An older build
# imports fine and then fails deep in a later cell on a key it never returned, which reads as
# a bug in the notebook. The version string is static, so the commit is what distinguishes
# them. A local checkout is left alone: it is someone working on the package.
_kind, _have = _installed()
_want = _latest()
#: Whether a previous cell already imported it. Reinstalling cannot replace a module object
#: that is already in memory, and the commit check above reads the disk, so "current" says
#: nothing about what this kernel is actually running.
_already = "ppropt" in sys.modules

if _kind == "local":
    _msg = "local checkout in use, left alone"
elif _kind == "absent" or (_kind == "github" and _want and _have != _want):
    token = None
    try:
        from google.colab import userdata          # private-repo fallback
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = (f"git+https://{token}@github.com/{REPO}.git" if token
           else f"git+https://github.com/{REPO}.git")
    if _kind == "absent":
        %pip install --quiet $url
        import ppropt                               # fail here, not three cells later
        _msg = f"installed ppropt @ {(_want or '?')[:7]}"
    elif "ppropt" in sys.modules:
        # Already imported: reinstalling cannot replace the module object in memory, so the
        # only honest thing to report is that a restart is still required.
        %pip install --quiet --force-reinstall --no-deps $url
        _msg = (f"updated to {(_want or '?')[:7]} — now use Runtime > Restart session and "
                f"run this cell again; the old copy is still loaded until you do")
    else:
        # Dependencies are already present; a newly added one fails loudly on import.
        %pip install --quiet --force-reinstall --no-deps $url
        import ppropt
        _msg = f"updated to {(_want or '?')[:7]}"
elif _kind == "other" or not _have:
    _msg = "ppropt installed, but not from git, so it cannot be checked against main"
elif _want is None:
    _msg = f"ppropt @ {_have[:7]}, could not reach GitHub to check it is current"
else:
    _msg = f"ppropt @ {_have[:7]}, current with {REPO}@main"
    if _already:
        _msg += ("  —  already imported in this session; if it was updated since, "
                 "Runtime > Restart session")

from IPython.display import HTML, display
display(HTML(
    f'<div style="border-left:3px solid #1a7f5a;padding:.5em .9em;'
    f'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">{_msg}</div>'))

In [ ]:
#@title Setup check - is the loaded build current? {display-mode: "form"}
#@markdown Checks if this notebook can run on the copy of PPR-opt this session has loaded.
#@markdown
#@markdown **Green?** Carry on to step 1. 
#@markdown **Red?** **Runtime → Restart session**, then run
#@markdown Setup again - reinstalling alone will not help, because the old copy stays in
#@markdown memory until the session restarts.
from IPython.display import HTML, display

# synthesis_readiness arrived with the readiness verdict, in the same change as the
# synthesis_ready key step 2 reads, so its absence is exactly the staleness that matters.
try:
    from ppropt.design import synthesis_readiness
    _stale = None
except ImportError as exc:
    _stale = str(exc)

if _stale:
    display(HTML(
        '<div style="border-left:3px solid #a8402c;padding:.6em 1em;font-size:13px;'
        'line-height:1.55">This kernel has an older PPROPT loaded.<br><code>'
        + _stale + '</code><br><br>Run <b>Runtime &gt; Restart session</b>, then run the '
        'setup cell again. Reinstalling alone will not help: the old copy stays in memory '
        'until the kernel restarts.</div>'))
else:
    display(HTML(
        '<div style="border-left:3px solid #1a7f5a;padding:.6em 1em;'
        'font-family:ui-monospace,monospace;font-size:13px;opacity:.85">'
        'loaded build carries the synthesis-readiness verdict</div>'))

## 1 · Your Target and Host

Fill this in and the rest of the notebook follows. Nothing below needs editing.

In [ ]:
#@title Edit your settings {display-mode: "form"}
#@markdown # A · What should it bind?
#@markdown ---
#@markdown The RNA sequence your PPR will recognise. **Its length sets the architecture** —
#@markdown 9, 14 or 19 bases give a 9S, 14S or 19S protein.
#@markdown
#@markdown ⏱️ **The Length of your RNA sequence also sets how long Step 2 takes, and the difference is large.** 
#@markdown Measured once each on the project's own machine, screening off: **9 bases ≈ 0.5 s**,
#@markdown **14 bases ≈ 14 s** and **19 bases ≈ 3-4 minutes**. The codon optimiser has to route a
#@markdown longer coding sequence around more fixed junctions and more forbidden enzyme sites,
#@markdown so the cost climbs far faster than the length does. A longer sequence is not stuck - it is
#@markdown working.
#@markdown
#@markdown 💡 **Not sure which window of your gene to aim at?** Leave this field alone and use
#@markdown **Pick the target window for me** under *Optional extras*. Give it a region and it
#@markdown picks the window inside it with the least off-target evidence against it.
target_rna = "AAAAUGUGG"  #@param {type:"string"}

#@markdown # B · Where will it be expressed?
#@markdown ---
#@markdown Sets the codon usage and the genetic code. (Nuclear hosts use table 1, Chloroplasts table 11)
#@markdown
#@markdown ⚠️ **The two Chlamydomonas entries are not interchangeable.** The nucleus is
#@markdown GC-rich and prefers Leu `CTG`; the chloroplast is AT-rich and prefers Leu `TTA`.
#@markdown Using one for the other produces DNA that looks fine but works very bad.
organism = "c_reinhardtii_nuclear"  #@param ["c_reinhardtii_nuclear", "c_reinhardtii_chloroplast", "e_coli", "s_cerevisiae", "a_thaliana_nuclear", "n_tabacum_chloroplast"]

#@markdown ### Your own codon usage - optional
#@markdown Leave all three blank to use the host above. Highest filled one wins.
#@markdown
#@markdown **`codon_table_file`** - a `codon,frequency` CSV or a CDS FASTA. Upload it with
#@markdown the folder icon in the sidebar, then put the filename here.
#@markdown
#@markdown **`kazusa_taxid`** - any NCBI taxonomy ID Kazusa carries, e.g. `4577` for maize.
#@markdown Fetched and cached on first use.
#@markdown
#@markdown **`genetic_code_override`** - the genetic code for a table you supplied. Leave 0
#@markdown to inherit from the host; set 11 for anything organellar.
codon_table_file = ""  #@param {type:"string"}
kazusa_taxid = 0  #@param {type:"integer"}
genetic_code_override = 0  #@param {type:"integer"}

#@markdown # C · How will it be assembled?
#@markdown ---
#@markdown Which Type IIS enzyme cuts the fragments out, and which published mis-ligation
#@markdown table scores the junctions. `BsaI-HFv2` and `BbsI-HF` are the two measured in
#@markdown Pryor *et al.* 2020 at 25 °C over 18 h.
assembly_enzyme = "BsaI"  #@param ["BsaI", "BbsI", "BsmBI", "SapI"]
ligation_table = "BsaI-HFv2"  #@param ["BsaI-HFv2", "BbsI-HF"]

#@markdown ### Which enzyme sites must be absent
#@markdown `assembly` — this assembly's own chemistry only &nbsp;·&nbsp;
#@markdown `igem_rfc1000` — adds SapI, required by iGEM's Type IIS standard &nbsp;·&nbsp;
#@markdown `moclo_compat` — adds BsmBI to keep later MoClo levels open, a preference that can
#@markdown make some junctions infeasible.
#@markdown
#@markdown **`extra_blacklist`** — anything else this experiment needs kept clear,
#@markdown comma-separated. Any name Biopython knows, for example `EcoRI, BamHI, HindIII, NotI`.
enzyme_profile = "igem_rfc1000"  #@param ["assembly", "igem_rfc1000", "moclo_compat"]
extra_blacklist = ""  #@param {type:"string"}

#@markdown ### Destination vector
#@markdown Or type your own acceptor overhangs below as `5prime,3prime` coding sites, which
#@markdown override the level. The 3' entry is the **coding site**; the enzyme leaves its
#@markdown reverse complement.
destination_level = "level0"  #@param ["level_minus1", "level0", "level1"]
custom_destination = ""  #@param {type:"string"}

#@markdown # D · Anything else
#@markdown ---
#@markdown **`n_fragments`** — how many pieces to split the gene into. Leave at 0 to let the
#@markdown length decide; set it only if your vendor has an awkward limit.
#@markdown
#@markdown **`seed`** — the same seed always gives the same design.
#@markdown
#@markdown **`write_files`** writes the design files to disk, and **`check_offtarget`** checks
#@markdown the target against the host genome.
n_fragments = 0  #@param {type:"integer"}
seed = 42  #@param {type:"integer"}
write_files = True  #@param {type:"boolean"}
check_offtarget = True  #@param {type:"boolean"}

## 2 · Design it

Run it. Nothing to edit here.

In [ ]:
#@title Run the design {display-mode: "form"}
#@markdown Builds the protein, decides where to cut it into fragments, and chooses DNA
#@markdown spellings that avoid awkward sequence. Nothing to edit.
#@markdown
#@markdown ⏱️ This is the slow step. A 9-base target takes about a second; a 19-base one can
#@markdown take several minutes. It has not hung.
from ppropt import DESTINATION_OVERHANGS, design_oneshot, table_from_kazusa
from IPython.display import HTML, display

# Resolved once and reused, so every design cell below uses the same table. The library
# cell passed codon_table=None and fell back to the dropdown, which meant typing a taxid
# here gave that host's design and a Chlamydomonas library two cells later.
chosen_codon_table = (codon_table_file or
                      (table_from_kazusa(kazusa_taxid) if kazusa_taxid else None))
chosen_table_source = (f"file {codon_table_file}" if codon_table_file
                       else f"Kazusa taxid {kazusa_taxid}" if kazusa_taxid
                       else organism)

result = design_oneshot(
    target_rna,
    organism=organism,
    codon_table=chosen_codon_table,
    genetic_code=genetic_code_override or None,
    enzyme_profile=enzyme_profile,
    extra_blacklist=extra_blacklist,
    enzyme=assembly_enzyme,
    matrix=ligation_table,
    destination=(tuple(s.strip().upper() for s in custom_destination.split(",")[:2])
                 if custom_destination else DESTINATION_OVERHANGS[destination_level]),
    n_fragments=n_fragments or None,
    seed=seed,
    check_offtarget=check_offtarget,
    outdir="ppropt_output" if write_files else None,
)

qc = result["qc"]

# tests/test_display_verdict.py renders this card from the cell's tail alone, with only
# `result` in scope, so the card must not depend on names defined beside the design call
# above. In the notebook this finds the real value; under the test it falls back.
table_source = globals().get("chosen_table_source", "as supplied")
TONE = {"PASS": "#1a7f5a", "WARNING": "#9a6b1f", "FAIL": "#a8402c"}

# The badge is the SYNTHESIS verdict, not the coding-span QC. It used to read the CDS
# status, so a design the final-artifact check rejects showed a green PASS and the
# rejection appeared nowhere on the card.
ready = result["synthesis_ready"]
verdict = "READY TO ORDER" if ready else "NOT ORDERABLE"
tone = TONE["PASS"] if ready else TONE["FAIL"]
why = result.get("not_ready_because") or []


def _stat(label, value, hint=""):
    return (
        f'<div style="padding:.55em .9em .6em;border-left:1px solid rgba(128,145,138,.35)">'
        f'<div style="font-size:10.5px;letter-spacing:.09em;text-transform:uppercase;'
        f'opacity:.6">{label}</div>'
        f'<div style="font-size:19px;font-weight:600;font-variant-numeric:tabular-nums;'
        f'margin-top:.15em">{value}</div>'
        f'<div style="font-size:11.5px;opacity:.6">{hint}</div></div>')


# Labelled, because an unlabelled TNTNND... under "coding sequence" reads as the sequence.
ppr_code = result["ppr_code"]
if len(ppr_code) > 22:
    ppr_code = ppr_code[:22] + "…"
ppr_code = f"PPR code {ppr_code}"

cards = "".join([
    _stat("codon table", table_source,
          "the dropdown is ignored when a file or taxid is set"),
    _stat("architecture", result["architecture"], f'{len(result["protein"])} aa protein'),
    _stat("coding sequence", f'{len(result["cds"])} nt', ppr_code),
    _stat("fragments", len(result["oligos"]),
          "cuts at " + ", ".join(str(c) for c in result["cuts"])),
    _stat("fidelity", f'{result["fidelity"]:.3f}', "predicted, not measured"),
    _stat("GC, coding span", f'{qc["gc_pct"]:.1f}%',
          f'CDS windows {qc["gc_window_min"]:.0f}–{qc["gc_window_max"]:.0f}%'),
    _stat("repeats, coding span", f'{qc["repeated_kmer_fraction"]:.1%}',
          f'longest {qc["longest_repeat"]} nt'),
    _stat("coding-span QC", qc["status"], "the CDS only"),
    _stat("final artifacts", result["final_artifacts"]["status"],
          "the sequences that would be ordered"),
])

# The reason the design was refused, in the place a reader looks first.
refusal = "" if ready else (
    f'<div style="border-left:3px solid {TONE["FAIL"]};padding:.6em .9em;'
    f'margin-top:.7em;font-size:13px;line-height:1.5">'
    f'<b>This design must not be ordered.</b> The coding sequence passed its own checks; '
    f'the sequences that would actually be ordered did not.<ul style="margin:.4em 0 0;'
    f'padding-left:1.2em">'
    + "".join(f"<li>{w}</li>" for w in why)
    + '</ul><div style="margin-top:.5em;opacity:.75">Files are still written for '
      'inspection, under <code>diagnostic_</code> keys with <code>REJECTED</code> in '
      'every filename. They are not an order.</div></div>')

warn = "".join(
    f'<div style="border-left:3px solid {TONE["WARNING"]};padding:.5em .9em;'
    f'margin-top:.7em;font-size:13px;line-height:1.5">{w}</div>'
    for w in result["warnings"])

RULE = "1px solid rgba(128,145,138,.35)"
card = (
    f'<div style="font-family:ui-sans-serif,system-ui,sans-serif;border:{RULE};'
    f'border-radius:5px;overflow:hidden;max-width:920px">'
    f'<div style="display:flex;align-items:center;gap:.8em;padding:.7em 1em;'
    f'border-bottom:{RULE}">'
    f'<span style="font-family:ui-monospace,monospace;font-size:16px;font-weight:600">'
    f'{result["target_rna"]}</span>'
    f'<span style="background:{tone};color:#fff;font-size:11px;font-weight:700;'
    f'letter-spacing:.07em;padding:.2em .7em;border-radius:99px">{verdict}</span>'
    f'</div>'
    f'<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(140px,1fr))">'
    f'{cards}</div></div>{refusal}{warn}')

display(HTML(card))

## 3 · The fragments

One row per fragment. **A row here is not an order:** whether these may be ordered is the
SYNTHESIS verdict on the result card above. `oh5` and `oh3` are the four-base Golden Gate overhangs that
join each fragment to its neighbours.

In [ ]:
#@title Draw the assembly {display-mode: "form"}
from ppropt.viz import assembly_svg

display(HTML(assembly_svg(
    result["oligos"],
    title=f'{result["target_rna"]}  ({result["architecture"]})',
    orderable=result["synthesis_ready"],
    verdict=("Orderable under the declared software checks."
             if result["synthesis_ready"]
             else "NOT ORDERABLE - " + (result["not_ready_because"] or ["refused"])[0]))))

In [ ]:
#@title Show the fragments {display-mode: "form"}
cols = {"fragment_id": "fragment", "assembly_order": "order", "aa_length": "residues",
        "oligo_length": "oligo nt", "oh5_coding_site_5to3": "oh5",
        "oh3_coding_site_5to3": "oh3"}
table = result["oligos"][list(cols)].rename(columns=cols)

# pandas' .style needs jinja2, which Colab has but a bare local environment may not.
# Test that directly rather than catching the AttributeError pandas raises, which would also
# swallow real errors. Falls back to the plain frame: cosmetics should never break a cell.
try:
    import jinja2  # noqa: F401
    display(table.style.hide(axis="index").set_properties(
        subset=["oh5", "oh3"], **{"font-family": "ui-monospace, monospace"}).set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("font-size", "11px"),
                                         ("letter-spacing", ".07em"),
                                         ("text-transform", "uppercase"),
                                         ("opacity", ".65"), ("padding", ".4em .9em")]},
            {"selector": "td", "props": [("padding", ".4em .9em"),
                                         ("font-variant-numeric", "tabular-nums")]}]))
except ImportError:
    display(table)

## 4 · Take the files

The order CSV, the oligos as FASTA, the assembled gene, and an annotated GenBank record —
every PPR repeat labelled with the base it reads — that opens in Benchling or SnapGene.

In [ ]:
#@title Download the files {display-mode: "form"}
import os
from IPython.display import HTML, display

if not result["paths"]:
    display(HTML('<div style="opacity:.7">Set <code>write_files</code> to True in the '
                 'parameters cell and re-run.</div>'))
else:
    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    LABEL = {"oligo_csv": "Order sheet (CSV)", "oligo_fasta": "Oligos (FASTA)",
             "gene_fasta": "Assembled gene (FASTA)", "genbank": "Annotated GenBank",
             # A rejected design exports under these keys. The label has to say so, or a
             # REJECTED filename in a table headed "Order sheet" reads as an order.
             "diagnostic_oligo_csv": "DIAGNOSTIC ONLY - rejected oligos (CSV)",
             "diagnostic_oligo_fasta": "DIAGNOSTIC ONLY - rejected oligos (FASTA)",
             "diagnostic_gene_fasta": "DIAGNOSTIC ONLY - rejected gene (FASTA)",
             "diagnostic_genbank": "DIAGNOSTIC ONLY - rejected GenBank"}
    if not result["synthesis_ready"]:
        display(HTML(
            '<div style="border-left:3px solid #a8402c;padding:.5em .9em;'
            'margin-bottom:.6em;font-size:13px;line-height:1.5">These files are for '
            '<b>inspection only</b>. This design did not pass the final-artifact check, '
            'so no order sheet was written.</div>'))
    rows = "".join(
        f'<tr><td style="padding:.35em .9em">{LABEL.get(k, k)}</td>'
        f'<td style="padding:.35em .9em;font-family:ui-monospace,monospace;font-size:12px;'
        f'opacity:.7">{os.path.basename(p)}</td>'
        f'<td style="padding:.35em .9em;text-align:right;font-variant-numeric:tabular-nums;'
        f'opacity:.7">{os.path.getsize(p):,} B</td></tr>'
        for k, p in result["paths"].items())
    display(HTML(f'<table style="font-family:ui-sans-serif,system-ui,sans-serif;'
                 f'font-size:13px;border-collapse:collapse">{rows}</table>'))

    if colab_files:
        for p in result["paths"].values():
            colab_files.download(p)

---

## Optional extras

*Nothing below is needed to place an order.* Use the arrow beside this heading to collapse
the whole section, or open any single item.

### Pick the target window for me

Give this a **region** — a UTR, a coding span, any stretch you are willing to target anywhere
within — and it chooses the window inside it with the least off-target evidence against it,
instead of you picking one by hand.

**Why a region beats a typed target.** A region is longer than the site a PPR reads, so most of
it is a choice you would otherwise be making silently. On a 200-base stretch of `rbcL`, with the
whole feature declared intended as this cell declares it, **99 of the 192 possible 9-base
windows** carry no other exact occurrence in the annotation, and **all 187 possible 14-base
windows** carry none. Choosing by hand throws that away.

**It also makes the host check mean something.** Given a bare target, the host scan cannot
separate the site you meant from a coincidence elsewhere in the genome — so a real match and
your own intended site look identical. Name the host gene your region came from and the intended
occurrence stops counting against you.

**What it does not do.** It ranks on other-occurrence count only. An exact match is a sequence
coincidence, not demonstrated binding, and nothing here measures affinity. It does not rank on
GC or on whether the resulting protein is easy to synthesise — those are properties of the
design that follows, not of the window.

By default this cell **reports and changes nothing**, so you can see what it would pick before
committing to it. Tick `apply_to_target` to set `target_rna`, then re-run Step 2.

In [ ]:
#@title Choose a window from a region {display-mode: "form"}
#@markdown Paste the region to search. Leave it blank and name a host gene below to use that
#@markdown gene's annotated span instead.
search_sequence = ""  #@param {type:"string"}
#@markdown Where the region came from. **Not inferred, because it changes what every occurrence
#@markdown means:** a designed sequence's intended site is absent from the host reference, while
#@markdown a native one's is present and will be found by any scan.
sequence_origin = "native"  #@param ["native", "designed", "unknown"]
#@markdown Which architectures to consider. Note that ranking puts fewest-occurrences first and
#@markdown **shorter second**, so whenever a clean 9-mer exists a combined option returns one;
#@markdown ask for 14 or 19 on its own to see what that architecture offers.
window_lengths = "9"  #@param ["9", "14", "19", "9, 14", "9, 14, 19"]
#@markdown The host gene the region comes from, used **only** when the origin is `native`:
#@markdown occurrences inside it are your intended site rather than off-targets. Blank declares
#@markdown nothing, which is safe but counts your own gene against you.
intended_gene = "rbcL"  #@param {type:"string"}
#@markdown Screen each window against the chloroplast annotation. Off searches the supplied
#@markdown region only.
screen_against_host = True  #@param {type:"boolean"}
#@markdown **Tick this to actually use the chosen window.** Left off, the cell reports what it
#@markdown would pick and changes nothing — so a *Run all* never silently redesigns for a
#@markdown target you did not choose.
apply_to_target = False  #@param {type:"boolean"}

import ppropt  # unguarded: a broken install should show its real traceback, not be excused

if not hasattr(ppropt, "select_window"):
    # Only the one case this guard is for: a build predating targeting.py. Anything else --
    # a missing Biopython, a half-installed package -- must raise and say what it is.
    print("This build of PPROPT has no window selection.")
    print("Use Runtime -> Restart session, re-run Setup, then run this cell again.")
    print("Nothing else is affected: target_rna from Step 1 is untouched.")
else:
    from ppropt import SearchSequence, loci_for_gene, select_window, window_report
    from ppropt.offtarget import load_transcripts

    transcripts = load_transcripts()
    lengths = tuple(int(n) for n in window_lengths.replace(",", " ").split())
    gene = intended_gene.strip()
    native = sequence_origin == "native"
    seq = search_sequence.strip()

    # Blank field plus a gene name: use that gene's span, so the cell is usable without
    # hunting for a sequence first.
    if not seq and gene:
        named = [t for t in transcripts if t.name == gene]
        if named:
            seq = named[0].sequence
            kind = named[0].kind
            print(f"using the annotated span of {gene}, {len(seq)} bases ({kind}).")
            # Stated from the feature, not assumed: TRANSCRIBED_TYPES also allows rRNA, tRNA,
            # ncRNA and friends, and calling a tRNA a coding span is simply false.
            print(f"Note this is the annotated {kind} span, not a UTR — the cached annotation")
            print("carries no UTRs.")
            if len(named) > 1:
                print(f"{len(named)} annotated features carry the name {gene}; took the first.")
            print()

    if not seq:
        if gene:
            # A typo here would otherwise read as "you gave me nothing".
            print(f"No annotated feature is named {gene}, so there is no span to search.")
            print("Check the spelling, or paste a region into search_sequence instead.")
        else:
            print("Nothing to search. Paste a region into search_sequence, or name a gene in")
            print("intended_gene to use its annotated span.")
    else:
        if gene and not native:
            print(f"origin is {sequence_origin!r}, so {gene} is NOT declared as intended.")
            print("Intent is only declared for a native region, because a designed sequence")
            print("has no intended site in a wild-type reference. If this region IS the host's")
            print("own, set sequence_origin to 'native' so your site stops counting against you.")
            print()
        pasted = bool(search_sequence.strip())
        loci = loci_for_gene(gene, transcripts) if (gene and native) else ()
        if gene and native and not loci:
            # Silently declaring nothing here rejects every window in the user's own gene and
            # then advises a longer window, which cannot help. Name the real cause.
            print(f"No annotated feature is named {gene} — the name is matched exactly and is")
            print("case-sensitive. Nothing is declared as intended, so occurrences inside your")
            print("own gene will count against every window. Check the spelling.")
            print()
        elif pasted and loci:
            print(f"Declaring {gene} as the intended site for the sequence you pasted. If that")
            print(f"region is not from {gene}, clear intended_gene — otherwise real off-target")
            print(f"occurrences inside {gene} are discounted as your own site.")
            print()
        source = SearchSequence(identifier=(gene if (gene and not pasted) else "supplied"),
                                sequence=seq, origin=sequence_origin)
        selection = select_window(
            source, lengths=lengths,
            transcripts=transcripts if screen_against_host else None,
            intended_loci=loci,
            reference="NC_005353.1" if screen_against_host else None)
        print(window_report(selection))
        print()
        if not selection.found:
            print("target_rna is unchanged — no window qualified, which is an answer and not")
            print("a failure. Try a longer window length, or a wider region.")
        elif apply_to_target:
            target_rna = selection.chosen.sequence
            if selection.chosen.intended:
                # Only now is this true: the host check reads target_rna, which just changed.
                print(f"Expect the host check below to report this window as occurring in "
                      f"{gene}.")
                print("That occurrence is the site you declared intended, and the count above")
                print("already excludes it. The two cells are not disagreeing: this one was")
                print("told where your site is and that one was not.")
                print()
            print(f"target_rna is now {target_rna}.")
            print()
            print("RE-RUN STEPS 2, 3 AND 4. Any fragment table or download from an earlier run")
            print("is for the previous target and is now out of date — including in a Run all,")
            print("where those steps ran before this cell.")
        else:
            was = globals().get("target_rna", "(not set yet — run Step 1)")
            print(f"target_rna is still {was} — nothing was changed. Tick apply_to_target")
            print(f"above and re-run this cell to design for {selection.chosen.sequence}.")

### Upload a codon table

Only if you want to supply your own codon usage from a file rather than typing a path.

In [ ]:
#@title Upload from your computer {display-mode: "form"}
#@markdown Optional. Run this only if you want to upload a codon table from your computer
#@markdown rather than type a path. It puts the file in the runtime and fills in the
#@markdown filename for you — then re-run the Design cell.
try:
    from google.colab import files as _f
    _up = _f.upload()
    if _up:
        codon_table_file = list(_up)[0]
        print(f"using {codon_table_file}")
except ImportError:
    print("not running on Colab — put the file beside the notebook and give its path "
          "in codon_table_file instead.")

### Does this target already occur in the host?

This looks for your target as an **exact match** in the *Chlamydomonas* chloroplast reference.
A match means the host already carries that sequence — a reason to look closer, **not** a
prediction that your PPR will bind there. Nothing here measures affinity.

**Only transcripts can matter.** A match in non-transcribed DNA is not an RNA target, and
neither is a reverse-complement match — the transcript from that locus carries the other
sequence. Both tiers are printed separately.

**What was searched.** 109 annotated features of one reference. Nine gene names appear
**twice** — `psbA` among them, as two identical copies at different places in the genome — so
a gene name does not identify a single location, and the CDS entries are coding spans rather
than UTR-inclusive transcripts. Your construct and the other members of your library are
**not** searched; for those, see the cross-talk cells below. Background rates and how they
were measured are in `PPROPT_FOR_THE_WET_LAB.md`.

> The target this notebook ships with, `AAAAUGUGG`, **does** occur in the host, deliberately,
> so a first run shows you what a match looks like rather than a blank result.

In [ ]:
#@title Run the host check {display-mode: "form"}
from ppropt.offtarget import architecture_advice, load_genome, load_transcripts, report, scan

genome = load_genome()
transcripts = load_transcripts()

# The answer first. A background-rate table printed above the result made readers ask what
# the statistic was for before they had seen whether their own target was flagged.
print(report([scan(target_rna, genome, transcripts)]))

# Length is the lever that fixes a flagged target, so it belongs beside the result. Name the
# quantity: this is a GENOMIC, both-strand figure, while the verdict above is about
# transcripts. Using one to dismiss the other is a confusion a reviewer already hit here.
expected = architecture_advice(genome).get(len(target_rna))
print()
if expected is not None:
    print(f"For scale: an average {len(target_rna)}-base sequence occurs about "
          f"{expected:.1f} time(s) by chance")
    print("in this genome, counting both strands of the DNA. That is a different quantity")
    print("from the transcript verdict above and does not explain it away.")
    print()
    print("Length is what buys specificity: across 200 random trials a 14-base target")
    print("occurred nowhere in this genome, in either tier.")

print()
print(f"host: Chlamydomonas reinhardtii chloroplast, {len(genome):,} bp, "
      f"{len(transcripts)} annotated transcripts")

---

### Do you already own the parts?

Everything above designs DNA to be **synthesised**. The GRASP authors also deposited a
42-plasmid kit, and a lab holding it can assemble many PPRs from parts it already has.

| route | what you supply | what it constrains |
|---|---|---|
| **de novo synthesis** | new DNA | nothing — full synonymous freedom |
| **GRASP module kit** | parts you already hold | fixed to the deposited modules |

An *n*-base target needs *n*+1 modules, and each internal join consumes one linker pair. The
kit holds exactly three, so 19 bases is the longest it can build — the cell below says so
plainly when asked for more.

> Module identity, overhangs and plate positions come from **Dennis et al. 2025 Table S1**.
> This selects PPR modules only, not the acceptor plasmids.

In [ ]:
#@title Check the kit {display-mode: "form"}
from ppropt import parts_report, select_parts

plan = select_parts(target_rna)
print(parts_report(plan))

---

### Designing a whole library

For a set of regulators what matters is **orthogonality**: PPRᵢ must bind UTRᵢ and not UTRⱼ.
The matrix below is the pairwise distance between targets — larger is better separated.

> This section is a **capability, not a validated result**. Every other part of this package
> is checked against a 200-design corpus; this one has unit tests only, because no ground
> truth for it exists.

In [ ]:
#@title Run the library design {display-mode: "form"}
#@markdown **These are three example targets, not derived from your target above.** Replace
#@markdown them with your own set. The point of this cell is the *pairwise* question — whether
#@markdown several regulators interfere with each other — which needs more than one target, so
#@markdown it cannot reuse the single target from the top of the notebook.
targets = "AAAAUGUGG, GCUAAAGAC, UUACACGUG"  #@param {type:"string"}

from ppropt import design_library

target_list = [t.strip().upper() for t in targets.split(",") if t.strip()]
lib = design_library(target_list, codon_table=chosen_codon_table, organism=organism,
                     enzyme_profile=enzyme_profile, seed=seed,
                     check_offtarget=check_offtarget,
                     outdir="ppropt_library" if write_files else None,
                     on_progress=lambda i, n, t: print(f"  {i}/{n}  {t}", flush=True))

print()
print(lib.summary())
print()
print(lib.crosstalk())

In [ ]:
#@title Show the library QC table {display-mode: "form"}
lib.qc_table()

### DNA shared between members

Cross-talk asks whether two PPRs could bind each other's **target**. This asks whether two
**genes** share enough identical DNA to recombine — a different question. Every member
carries the same scaffold, so the answer is never trivially no.

`diversify_library` gives each member its own synonymous encoding of that scaffold. On five
9S designs it took the longest shared stretch from 107 nt to 47 and cleared every pair over
the 50 nt threshold; at six members it reaches only 77 nt and leaves 2 of 15 pairs above it.

A shared stretch is a *necessary* substrate for recombination, never a prediction that it
will happen, and 50 nt is a rule of thumb rather than a measured constant for this host.

In [ ]:
#@title Measure shared DNA {display-mode: "form"}
print(lib.homology())

### Cross-talk, in two tiers

Sequence separation and *predicted binding* are different questions, so they stay apart:

| tier | what it is | status |
|---|---|---|
| **A — Hamming distance** | two targets differ in *k* of *n* positions | **the only thing that gates** |
| **B — predicted affinity** | the PPR designed for A, scored against B | an annotation; gates nothing |

Tier B needs a PPR specificity table. **PPROPT does not ship one** — the available table
carries no licence, and it comes from **P-type** experiments while this scaffold is
**S-type**. A high score means *look*, never *fail*.

Leave the path blank and you get tier A alone, which is what gates anyway.

In [ ]:
#@title Run the cross-talk check {display-mode: "form"}
ppr_score_table = ""  #@param {type:"string"}

from ppropt import compare_tiers, load_ppr_scores

scores = load_ppr_scores(ppr_score_table) if ppr_score_table.strip() else None
print(lib.crosstalk(scores=scores))

if scores:
    cmp = compare_tiers(lib.targets, scores)
    print()
    print(f"pairs flagged by separation:      {len(cmp['hamming_flagged'])}")
    print(f"pairs flagged by predicted affinity: {len(cmp['affinity_flagged'])}")
    print(f"the model points somewhere separation does not: {cmp['tiers_disagree']}")
    print()
    print("A disagreement is a reading recommendation, not a failed design —")
    print("tier A alone decides what this library accepts.")

---

## Going deeper — why this design, and not another

*You do not need this to order. Read it if a design looks surprising.*

The gene is cut into fragments, and each place two fragments rejoin is a **junction**. Every
junction needs a 4-base sticky end, the **overhang**, so the right two fragments anneal to
each other and to nothing else.

The constraint that makes this hard: the overhang is made of **your own coding sequence** at
that point, not a linker bolted on. So the only overhangs available are the ones synonymous
codons can spell there while leaving the protein unchanged.

Below, each junction lists what it could have used and what became of it — **selected**,
**considered** (feasible, but another scored at least as well), or **rejected** (no synonymous
arrangement avoids an excluded enzyme site, so that overhang is impossible here, not merely
worse).

In [ ]:
#@title Show the audit {display-mode: "form"}
audit = result["audit"]
print(audit.report())

rejected = audit.rejected
print(f"\n{len(rejected)} candidate overhang(s) ruled out entirely under "
      f"profile '{audit.enzyme_profile}'")
for d in rejected[:8]:
    print(f"    {d.sequence}  cut {d.junction_cut:>4d}   {d.reason}")
if len(rejected) > 8:
    print(f"    … and {len(rejected) - 8} more")
if not rejected:
    print("    (every achievable overhang was usable at every junction)")

---

## What each step does

| Step | What happens |
|---|---|
| **0 · Setup** | installs PPROPT from GitHub |
| **1 · Target and host** | your target RNA, codon table, enzyme and destination |
| **2 · Design it** | target to protein, then cuts and overhangs chosen **before** the coding sequence is optimised, so the optimiser cannot rewrite the bases the junctions depend on |
| **3 · Fragments** | the oligo table you order |
| **4 · Files** | FASTA, GenBank and the oligo CSV |
| *Optional extras* | **choosing the target window from a region**, host occurrence check, the deposited-kit route, a whole library, cross-talk |
| *Going deeper* | every overhang considered at every junction, and why each was kept or ruled out |

**Predicted fidelity** is from published ligation-count matrices (Pryor *et al.* 2020), not a measured efficiency in your hands. **QC** is a sequence-complexity check, not calibrated against vendor outcomes. No sequence from this project has been synthesised.

Building from the deposited 42-module kit instead? [PPROPT_inventory.ipynb](https://colab.research.google.com/github/SyedZainAliShah/clippr/blob/main/notebooks/PPROPT_inventory.ipynb)

---

## Words used here

Every term this notebook uses, in plain language. Nothing here is a claim about your
design — only what the word means.

| term | what it means |
|---|---|
| **PPR protein** | What this tool designs. A chain of near-identical **repeats**, each one gripping a single RNA base. |
| **target** | The RNA sequence you want the protein to bind. You choose it. |
| **repeat** | One unit of the protein. Two amino acids inside it decide which RNA base it grips, and that mapping is published, so the design is a lookup rather than a search. |
| **architecture (9S / 14S / 19S)** | How many repeats the protein has. It follows from your target's length: 9 bases gives 9S, 14 gives 14S, 19 gives 19S. |
| **codon** | Three DNA letters spelling one amino acid. Most amino acids have several possible codons, so the same protein can be written many different ways. |
| **codon optimisation** | Choosing among those spellings so the DNA is easier to manufacture and easier for your host to read. The protein does not change. |
| **coding sequence (CDS)** | The DNA that spells the protein, start to finish. |
| **fragment** | One of the pieces a gene is split into. Long stretches of DNA are unreliable to synthesise, so the gene is made in a few parts and joined. |
| **overhang** | Four unpaired bases at the end of a fragment. Only its intended partner matches it, so the pieces can join in one order and no other. |
| **`oh5` / `oh3`** | The overhang at a fragment's start and at its end. |
| **junction** | The point where two fragments meet, held together by their matching overhangs. |
| **Golden Gate assembly** | The method used to join the fragments: one enzyme cuts and the pieces are stitched together in a single reaction. |
| **BsaI** | That enzyme. It cuts *outside* the sequence it recognises, which is what lets the overhangs be any four bases you like. |
| **wrapper** | Fixed DNA added to each end of a fragment so the enzyme has something to cut. It is not part of your gene, but it **is** part of what you order. |
| **GC content** | The share of bases that are G or C. Very high or very low makes DNA hard to synthesise. |
| **oligo** | A piece of synthesised DNA. Here, one fragment. |
| **fidelity** | The predicted chance the pieces assemble in the intended order. It comes from published measurements of this enzyme, not from your bench. |
| **coding-span QC** | Checks run on the gene itself. |
| **final artifacts** | Checks run on the exact sequences you would send to a vendor, wrapper included. This is the one that decides orderable. |
| **orderable** | Passed every check this software declares. It is **not** vendor acceptance, and **not** a claim that the protein works. |
| **off-target** | Whether your target sequence also turns up in the host's own genome. An exact match is a reason to look closer, not a prediction that anything binds there. |
| **cross-talk** | In a library, whether one PPR might also bind another member's target. It is what stops several regulators being controlled independently. |
| **library** | Several PPRs designed together, so they can be checked against each other rather than one at a time. |